# Lab 2.1: Context Preservation Across Long Interactions

**What you'll build:** A customer support system that preserves critical case facts (amounts, order numbers, dates) across long conversations — and learn why progressive summarization destroys the details you need most.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — watch progressive summarization lose critical numbers | 5 min |
| 2 | The Right Way — structured case facts blocks preserve details | 5 min |
| 3 | Your Turn — build a context preservation system for a billing dispute | 10 min |
| 4 | Stress Test — prove your solution works across 20+ turns | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You're building a customer support agent that handles billing disputes. These conversations are long (15-25 turns) and involve specific numbers that must be accurate in the final resolution.

The challenge: as conversations grow, the model's ability to reference specific details from earlier turns degrades. Dollar amounts, order numbers, and dates get "summarized" into vague references.

We'll simulate this with a realistic billing dispute scenario.

In [ ]:
# A realistic billing dispute with specific numbers that must be preserved
CASE_FACTS = {
    "customer_id": "CUST-28491",
    "customer_name": "Sarah Chen",
    "order_number": "ORD-2024-78923",
    "original_amount": 1234.56,
    "disputed_amount": 489.99,
    "order_date": "2024-11-15",
    "dispute_date": "2024-12-02",
    "items_in_dispute": [
        {"sku": "WIDGET-A", "price": 299.99, "status": "returned"},
        {"sku": "GADGET-B", "price": 190.00, "status": "damaged_in_shipping"}
    ],
    "payment_method": "Visa ending 4821",
    "loyalty_tier": "Gold",
    "previous_disputes": 0
}

# Simulate a long conversation about this dispute
CONVERSATION_TURNS = [
    f"Hi, I'm calling about order {CASE_FACTS['order_number']}. I was charged ${CASE_FACTS['original_amount']} but two items had problems.",
    f"The WIDGET-A for ${CASE_FACTS['items_in_dispute'][0]['price']} was returned on November 20th and I still haven't gotten my refund.",
    f"Also the GADGET-B for ${CASE_FACTS['items_in_dispute'][1]['price']} arrived damaged. The box was crushed.",
    "I've been a loyal customer for 5 years. This is really frustrating.",
    "Can you check what happened with my return? The tracking shows it was received.",
    "I paid with my Visa ending in 4821. When will the refund show up?",
    "My total disputed amount is $489.99. I need this resolved today.",
    "What's your policy on damaged items? Do I need to send it back?",
    "OK so to summarize - I need a refund for the returned widget and a replacement or refund for the damaged gadget.",
    "Can you send me a confirmation email with all the details of what's being refunded?"
]

print(f"Case: {CASE_FACTS['customer_name']} disputing ${CASE_FACTS['disputed_amount']} on order {CASE_FACTS['order_number']}")
print(f"Conversation length: {len(CONVERSATION_TURNS)} customer turns")
print(f"\nCritical details to preserve:")
print(f"  Order: {CASE_FACTS['order_number']}")
print(f"  Total: ${CASE_FACTS['original_amount']}")
print(f"  Disputed: ${CASE_FACTS['disputed_amount']}")
for item in CASE_FACTS['items_in_dispute']:
    print(f"  Item: {item['sku']} @ ${item['price']} ({item['status']})")

---

## Phase 1: The Wrong Approach

Most systems rely on the conversation history alone. As the conversation grows, earlier details get pushed further from the model's "attention window." Let's simulate what happens when we ask the model to write a resolution summary after a long conversation.

In [ ]:
# Build a conversation history WITHOUT structured context
def build_naive_conversation():
    """Build a conversation with only the raw chat history."""
    system = "You are a customer support agent. Help resolve billing disputes."
    
    messages = []
    for i, turn in enumerate(CONVERSATION_TURNS):
        messages.append({"role": "user", "content": turn})
        # Simulate agent responses (abbreviated)
        if i < len(CONVERSATION_TURNS) - 1:
            messages.append({"role": "assistant", "content": f"I understand. Let me look into that for you."})
    
    # Final ask: write a resolution summary
    messages.append({"role": "assistant", "content": "I've reviewed everything. Let me prepare a resolution summary."})
    messages.append({"role": "user", "content": 
        "Please write the resolution summary email with ALL specific details: "
        "order number, exact dollar amounts for each item, item SKUs, payment method, "
        "and what action is being taken for each item."})
    
    return system, messages

system, messages = build_naive_conversation()

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=system,
    messages=messages
)

naive_summary = response.content[0].text
print("=== NAIVE APPROACH: Resolution Summary ===")
print(naive_summary)

In [ ]:
# Check if critical details were preserved
def check_details(summary, facts):
    """Check which critical details appear in the summary."""
    checks = {
        "order_number": facts["order_number"] in summary,
        "original_amount": str(facts["original_amount"]) in summary,
        "disputed_amount": str(facts["disputed_amount"]) in summary,
        "widget_price": str(facts["items_in_dispute"][0]["price"]) in summary,
        "gadget_price": str(facts["items_in_dispute"][1]["price"]) in summary,
        "widget_sku": facts["items_in_dispute"][0]["sku"] in summary,
        "gadget_sku": facts["items_in_dispute"][1]["sku"] in summary,
        "payment_method": "4821" in summary,
    }
    return checks

naive_checks = check_details(naive_summary, CASE_FACTS)
print("=== DETAIL PRESERVATION CHECK ===")
for detail, found in naive_checks.items():
    status = "FOUND" if found else "MISSING"
    print(f"  {detail}: {status}")

found_count = sum(naive_checks.values())
print(f"\nPreserved {found_count}/{len(naive_checks)} critical details")
if found_count < len(naive_checks):
    print("\nThe naive approach lost details that were in the conversation.")
    print("This is the context degradation problem — details get vague over long conversations.")

### Why did that fail?

- **"Lost in the middle" effect.** In long conversations, details from the middle turns (where the specific amounts were mentioned) get less attention than the beginning and end.
- **No structured extraction.** Critical numbers were scattered across multiple turns as natural language — the model has to reconstruct them from conversation flow.
- **Progressive degradation.** Each turn pushes earlier details further from the model's attention. By turn 10, turn 2's specific dollar amount is far from the model's focus.
- **Summarization instinct.** The model naturally "summarizes" by abstracting: "$299.99 for WIDGET-A" becomes "the returned item" or "the widget."

---

## Phase 2: The Right Approach

The fix: **structured case facts blocks** injected into the system prompt. Instead of relying on the model to extract details from conversation history, we maintain a structured record that persists regardless of conversation length.

In [ ]:
def build_structured_conversation():
    """Build a conversation WITH structured case facts in system prompt."""
    
    # Key technique: structured case facts block in system prompt
    case_facts_block = json.dumps(CASE_FACTS, indent=2)
    
    system = f"""You are a customer support agent resolving billing disputes.

=== CASE FACTS (always reference these exact values) ===
{case_facts_block}
=== END CASE FACTS ===

CRITICAL: When writing summaries, resolutions, or emails, always use the exact
values from the CASE FACTS block above. Never paraphrase dollar amounts, order
numbers, or SKUs. These are the authoritative values for this case."""
    
    messages = []
    for i, turn in enumerate(CONVERSATION_TURNS):
        messages.append({"role": "user", "content": turn})
        if i < len(CONVERSATION_TURNS) - 1:
            messages.append({"role": "assistant", "content": f"I understand. Let me look into that for you."})
    
    messages.append({"role": "assistant", "content": "I've reviewed everything. Let me prepare a resolution summary."})
    messages.append({"role": "user", "content": 
        "Please write the resolution summary email with ALL specific details: "
        "order number, exact dollar amounts for each item, item SKUs, payment method, "
        "and what action is being taken for each item."})
    
    return system, messages

system, messages = build_structured_conversation()

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=system,
    messages=messages
)

structured_summary = response.content[0].text
print("=== STRUCTURED APPROACH: Resolution Summary ===")
print(structured_summary)

In [ ]:
# Compare both approaches
structured_checks = check_details(structured_summary, CASE_FACTS)

print("=" * 55)
print("COMPARISON: NAIVE vs STRUCTURED")
print("=" * 55)
print(f"{'Detail':<25} {'Naive':>12} {'Structured':>12}")
print(f"{'-'*25} {'-'*12} {'-'*12}")
for detail in naive_checks:
    n = "FOUND" if naive_checks[detail] else "MISSING"
    s = "FOUND" if structured_checks[detail] else "MISSING"
    print(f"{detail:<25} {n:>12} {s:>12}")

naive_score = sum(naive_checks.values())
struct_score = sum(structured_checks.values())
print(f"\n{'Total preserved':<25} {naive_score:>8}/{len(naive_checks)} {struct_score:>8}/{len(structured_checks)}")

if struct_score > naive_score:
    print("\nStructured case facts preserved more details than raw conversation history.")

### Side-by-Side Comparison

| Aspect | Naive (conversation only) | Structured (case facts block) |
|--------|--------------------------|-------------------------------|
| **Where details live** | Scattered across multiple turns | Single JSON block in system prompt |
| **Dollar amounts** | May be paraphrased or lost | Exact values preserved |
| **Order numbers** | May be truncated or wrong | Always exact |
| **After 20+ turns** | "The refund we discussed" | Still references $489.99 |
| **Resilience** | Degrades with conversation length | Constant — system prompt always present |

---

## Phase 3: Your Turn

Build a context preservation system for a different scenario: a technical support case with multiple devices, serial numbers, and firmware versions.

**Your goal:** Write a system prompt with structured case facts that preserves all device details across a 10-turn conversation.

In [ ]:
TECH_CASE = {
    "ticket_id": "TKT-2024-55012",
    "customer": "Marcus Johnson",
    "site": "Building C, Floor 3",
    "devices": [
        {"serial": "SN-AP-33921", "model": "AirPoint Pro 6", "firmware": "v3.2.1", "issue": "intermittent_drops"},
        {"serial": "SN-AP-33922", "model": "AirPoint Pro 6", "firmware": "v3.1.8", "issue": "no_5ghz"},
        {"serial": "SN-SW-10445", "model": "NetSwitch 48", "firmware": "v2.8.0", "issue": "none"}
    ],
    "network_segment": "VLAN 42 (Guest WiFi)",
    "affected_users": 150,
    "sla_deadline": "2024-12-05T17:00:00Z"
}

TECH_CONVERSATION = [
    f"Hi, we have WiFi issues on {TECH_CASE['site']}. About {TECH_CASE['affected_users']} users affected.",
    f"We have two access points: {TECH_CASE['devices'][0]['serial']} and {TECH_CASE['devices'][1]['serial']}. Both AirPoint Pro 6.",
    f"The first AP ({TECH_CASE['devices'][0]['serial']}) has intermittent drops. It's running firmware {TECH_CASE['devices'][0]['firmware']}.",
    f"The second one ({TECH_CASE['devices'][1]['serial']}) can't broadcast 5GHz at all. It's on older firmware {TECH_CASE['devices'][1]['firmware']}.",
    f"The switch ({TECH_CASE['devices'][2]['serial']}) seems fine but I wanted to mention it.",
    f"This is all on {TECH_CASE['network_segment']}. We have a corporate event Friday so we need this fixed by then.",
    "We've already tried rebooting both APs. No change.",
    "Can you check if there's a firmware update that addresses the 5GHz issue?",
    "Also, should we update both APs to the same firmware version for consistency?",
    "Please send me a summary of the plan with all device details so I can share with my manager."
]

# Ground truth: details that must appear in the final summary
REQUIRED_DETAILS = {
    "ticket_id": TECH_CASE["ticket_id"],
    "ap1_serial": TECH_CASE["devices"][0]["serial"],
    "ap2_serial": TECH_CASE["devices"][1]["serial"],
    "ap1_firmware": TECH_CASE["devices"][0]["firmware"],
    "ap2_firmware": TECH_CASE["devices"][1]["firmware"],
    "switch_serial": TECH_CASE["devices"][2]["serial"],
    "vlan": "VLAN 42",
    "affected_users": str(TECH_CASE["affected_users"]),
}

print("Tech case loaded.")
print(f"  Ticket: {TECH_CASE['ticket_id']}")
print(f"  Devices: {len(TECH_CASE['devices'])}")
print(f"  Required details to preserve: {len(REQUIRED_DETAILS)}")
print(f"\nWrite your system prompt in the next cell.")

In [ ]:
# =============================================================
# TODO: Write a system prompt with structured case facts
# =============================================================
#
# Requirements:
# - Include a structured case facts block (JSON) in the system prompt
# - Instruct the model to always reference exact values from the facts block
# - Make sure all device serial numbers, firmware versions, and network
#   details will be preserved in any summary the model writes
#
# Think about:
# - Where should the case facts block go in the system prompt?
# - What instruction tells the model to use exact values vs paraphrasing?

YOUR_SYSTEM_PROMPT = """
# TODO: Replace with your system prompt containing structured case facts.
# Use the CASE FACTS pattern from Phase 2.
You are a technical support agent.
"""

# Build conversation and run
messages = []
for i, turn in enumerate(TECH_CONVERSATION):
    messages.append({"role": "user", "content": turn})
    if i < len(TECH_CONVERSATION) - 1:
        messages.append({"role": "assistant", "content": "Got it, I'm noting that down."})

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system=YOUR_SYSTEM_PROMPT,
    messages=messages
)

your_summary = response.content[0].text
print("=== YOUR SYSTEM: Resolution Summary ===")
print(your_summary)

In [ ]:
# =============================================================
# Checker: validates your solution
# =============================================================

your_checks = {}
for detail_name, detail_value in REQUIRED_DETAILS.items():
    your_checks[detail_name] = detail_value in your_summary

print("=== YOUR SCORECARD ===")
for detail, found in your_checks.items():
    status = "FOUND" if found else "MISSING"
    print(f"  {detail}: {status}")

found_count = sum(your_checks.values())
total = len(your_checks)

print(f"\n  Preserved: {found_count}/{total}")

if found_count == total:
    print("\n  PASSED — all critical details preserved!")
    print("  Your structured case facts block successfully kept all device details accessible.")
else:
    missing = [d for d, f in your_checks.items() if not f]
    print(f"\n  MISSING: {missing}")
    print("  Revise your system prompt — ensure the case facts block contains all device details.")

### Success Criteria

The checker tests:

1. **All serial numbers** — each device's serial must appear exactly in the summary
2. **Firmware versions** — both AP firmware versions must be present
3. **Network details** — VLAN 42 and 150 users must appear
4. **Ticket ID** — must reference the exact ticket number

If details are missing, your case facts block may:
- Not contain all the device information
- Not instruct the model to use exact values
- Be placed in a position where the model doesn't reference it

---

## Phase 4: Stress Test

Run your system prompt multiple times to verify consistent detail preservation.

In [ ]:
print("Running your system prompt 5 times...\n")

results = []
for run in range(5):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        system=YOUR_SYSTEM_PROMPT,
        messages=messages
    )
    summary = response.content[0].text
    checks = {d: (v in summary) for d, v in REQUIRED_DETAILS.items()}
    found = sum(checks.values())
    results.append(found)
    missing = [d for d, f in checks.items() if not f]
    print(f"  Run {run+1}: {found}/{len(REQUIRED_DETAILS)} details preserved", end="")
    if missing:
        print(f" (missing: {missing})")
    else:
        print(" (perfect)")

print(f"\n=== CONSISTENCY ===")
print(f"Scores across 5 runs: {results}")
if all(r == len(REQUIRED_DETAILS) for r in results):
    print("Perfect consistency — your case facts block works reliably.")
else:
    print(f"Inconsistent — some runs lost details. Strengthen your system prompt.")

### Key Takeaways

1. **Progressive summarization loses specifics.** Dollar amounts, serial numbers, and dates degrade to vague references as conversations grow.
2. **The "lost in the middle" effect is real.** Information in the middle of long contexts gets less attention than content at the beginning and end.
3. **Structured case facts blocks are the fix.** A JSON block in the system prompt persists regardless of conversation length.
4. **Position matters.** Place case facts at the top of the system prompt (beginning) for maximum attention, not buried in the middle.
5. **Explicit instructions help.** Tell the model to "use exact values from the CASE FACTS block" — don't assume it will.